In [3]:
import pandas as pd
import json

df = pd.read_parquet('data/medical_consultation/train_tr1.parquet')

# Convert reward_model, prompt, extra_info to dict by json.loads
for i in range(len(df)):
    row_dict = df.iloc[i].to_dict()
    # Assuming the columns exist and contain valid JSON strings
    # row_dict['reward_model'] = json.loads(row_dict['reward_model'])
    # row_dict['prompt'] = json.loads(row_dict['prompt'])
    # row_dict['extra_info'] = json.loads(row_dict['extra_info'])
    row_dict['extra_info']['index'] = i
    # Update the DataFrame with the converted dictionaries
    df.iloc[i] = row_dict

df.to_parquet("data/medical_consultation/train_tr1.parquet")

In [4]:
df.prompt.values[0]

array([{'content': "<|im_start|>user\nYou are an experienced doctor who needs to provide professional diagnosis and advice to patients through consultation. Please listen carefully to the patient's description, ask targeted questions, and collect sufficient information before giving a diagnosis and treatment recommendation.\n\nQuick Guide\nObjectives:\n1. Obtain key information through effective questioning, each round of questions should be modified based on the previous round's content, meaning you shouldn't ask similar questions.\n2. Comprehensively analyze the patient's condition to provide an accurate diagnosis and appropriate treatment recommendations.\n\nRules:\n1. You can only choose one of the options to respond, you cannot both answer questions and provide a diagnosis simultaneously.\n2. Absolutely do not repeat or ask questions similar or identical to those previously asked.\n\nResponse:\n<answer>If you believe there is insufficient information, please only ask one question,

In [5]:
df.reward_model.values[0]

{'ground_truth': {'diagnosis': 'Based on your symptoms, it appears you have enteritis, which is inflammation of the intestines.',
  'recommendation': 'I recommend trying Trimebutine to help regulate your gastrointestinal function. If your symptoms persist or worsen, please follow up for further evaluation. In the meantime, avoid spicy or greasy foods and stay hydrated.'},
 'patient_information': array([{'doctor_question': array(['Hello, how long has this been going on?'], dtype=object), 'patient_response': array(['About two or three days.',
               "It's a dull pain that comes and goes."], dtype=object)}                                                                             ,
        {'doctor_question': array(['Have you taken any medication or had any tests done?'],
              dtype=object), 'patient_response': array(['No medication or tests.'], dtype=object)},
        {'doctor_question': array(['Are your bowel movements normal?'], dtype=object), 'patient_response': arra

In [7]:
import pandas as pd
import json

df = pd.read_parquet('data/medical_consultation/train_tr1.parquet')

# select the first 100 rows as val_tr1
val_tr1 = df.iloc[:100]
# select the rest as train_tr1
train_tr1 = df.iloc[100:]

# reset the index of val_tr1 and train_tr1
for i in range(len(val_tr1)):
    val_tr1.iloc[i]['extra_info']['index'] = i
for i in range(len(train_tr1)):
    train_tr1.iloc[i]['extra_info']['index'] = i

# save val_tr1 and train_tr1 to parquet
val_tr1.to_parquet("data/medical_consultation/val_tr1.parquet")
train_tr1.to_parquet("data/medical_consultation/train_tr1.parquet")

In [11]:
# 判断train_tr1的reward_model的patient_information中doctor_question和patient_response是否为列表，如果不是，则转换为列表
for i in range(len(train_tr1)):
    for j in range(len(train_tr1.iloc[i]['reward_model']['patient_information'])):
        if isinstance(train_tr1.iloc[i]['reward_model']['patient_information'][j]['doctor_question'], str):
            train_tr1.iloc[i]['reward_model']['patient_information'][j]['doctor_question'] = [train_tr1.iloc[i]['reward_model']['patient_information'][j]['doctor_question']]
            print("revise doctor_question")
        if isinstance(train_tr1.iloc[i]['reward_model']['patient_information'][j]['patient_response'], str):
            train_tr1.iloc[i]['reward_model']['patient_information'][j]['patient_response'] = [train_tr1.iloc[i]['reward_model']['patient_information'][j]['patient_response']]
            print("revise patient_response")

# train_tr1.to_parquet("data/medical_consultation/train_tr1.parquet")


In [23]:
# 判断train_tr1的reward_model的patient_information中doctor_question和patient_response是否为列表，如果不是，则转换为列表
import numpy as np
for i in range(len(val_tr1)):
    for j in range(len(val_tr1.iloc[i]['reward_model']['patient_information'])):
        assert isinstance(val_tr1.iloc[i]['reward_model']['patient_information'][j]['patient_response'], np.ndarray)
        if isinstance(val_tr1.iloc[i]['reward_model']['patient_information'][j]['doctor_question'], str):
            val_tr1.iloc[i]['reward_model']['patient_information'][j]['doctor_question'] = [val_tr1.iloc[i]['reward_model']['patient_information'][j]['doctor_question']]
            print("revise doctor_question")
        if isinstance(val_tr1.iloc[i]['reward_model']['patient_information'][j]['patient_response'], str):
            val_tr1.iloc[i]['reward_model']['patient_information'][j]['patient_response'] = [val_tr1.iloc[i]['reward_model']['patient_information'][j]['patient_response']]
            print("revise patient_response")

# val_tr1.to_parquet("data/medical_consultation/val_tr1.parquet")


AssertionError: 

In [24]:
val_tr1.iloc[i]['reward_model']['patient_information'][j]['patient_response']

In [13]:
def _calculate_lcs(str1: str, str2: str) -> str:
    """
    Calculate the longest common subsequence between two strings.
    """
    # Convert strings to lowercase for case-insensitive comparison
    str1 = str1.lower()
    str2 = str2.lower()
    
    # Get lengths of strings
    m = len(str1)
    n = len(str2)
    
    # Initialize LCS matrix
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    # Fill dp table
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if str1[i-1] == str2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
                
    # Backtrack to find LCS string
    lcs = []
    i, j = m, n
    while i > 0 and j > 0:
        if str1[i-1] == str2[j-1]:
            lcs.append(str1[i-1])
            i -= 1
            j -= 1
        elif dp[i-1][j] > dp[i][j-1]:
            i -= 1
        else:
            j -= 1
            
    return ''.join(reversed(lcs))

In [14]:
_calculate_lcs("小儿麻痹症", "小儿")

'小儿'